In [1]:
## Step 1: Load Configuration and Import Libraries

import json, os, shutil
from PIL import Image
import numpy as np
from tqdm.notebook import tqdm

with open('/kaggle/input/notebooks/kevinchovatiya/01-kaggle-data-setup/paths_config.json') as f:
    paths = json.load(f)

DATASET_PATH = paths['DATASET_PATH']
OUTPUT_PATH  = paths['OUTPUT_PATH']

print(f"Dataset path: {DATASET_PATH}")
print("✓ Libraries imported!")

Dataset path: /kaggle/input/datasets/shubhamgoel27/dermnet
✓ Libraries imported!


In [2]:

## Step 2: (Optional) Copy Dataset to Working Directory

# ========================================
# UNCOMMENT to copy dataset for modification
# ========================================
# WORKING_DATASET = '/kaggle/working/dermnet_working'
# if not os.path.exists(WORKING_DATASET):
#     print("Copying dataset to working directory (this takes a few minutes)...")
#     shutil.copytree(DATASET_PATH, WORKING_DATASET)
#     print(f"✓ Copied to: {WORKING_DATASET}")
#     # Update paths config to point to working copy
#     paths['DATASET_PATH'] = WORKING_DATASET
#     DATASET_PATH = WORKING_DATASET
#     with open('/kaggle/working/paths_config.json', 'w') as f:
#         json.dump(paths, f, indent=4)
# else:
#     print(f"Working dataset already exists: {WORKING_DATASET}")

print("Using dataset at:", DATASET_PATH)

Using dataset at: /kaggle/input/datasets/shubhamgoel27/dermnet


In [3]:
## Step 3: Detect Corrupted Images

def check_corrupted_images(base_path):
    print("Checking for corrupted images...\n")
    corrupted_files = []

    for split in ['train', 'test']:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            continue

        classes = [d for d in os.listdir(split_path)
                  if os.path.isdir(os.path.join(split_path, d))]

        for class_name in tqdm(classes, desc=f"Checking {split}"):
            class_path = os.path.join(split_path, class_name)
            images = [f for f in os.listdir(class_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            for img_name in images:
                img_path = os.path.join(class_path, img_name)
                try:
                    img = Image.open(img_path)
                    img.verify()
                    img = Image.open(img_path)
                    img.load()
                except Exception as e:
                    corrupted_files.append({
                        'path': img_path, 'class': class_name,
                        'split': split, 'error': str(e)
                    })

    if corrupted_files:
        print(f"\n⚠️  Found {len(corrupted_files)} corrupted images:")
        for file_info in corrupted_files:
            print(f"   • {file_info['path']}  —  {file_info['error']}")
    else:
        print("\n✓ No corrupted images found!")

    return corrupted_files

corrupted_files = check_corrupted_images(DATASET_PATH)

Checking for corrupted images...



Checking train:   0%|          | 0/23 [00:00<?, ?it/s]

Checking test:   0%|          | 0/23 [00:00<?, ?it/s]


✓ No corrupted images found!


In [4]:
## Step 4: Remove Corrupted Images (Optional)

def remove_corrupted_images(corrupted_files):
    if not corrupted_files:
        print("No corrupted images to remove.")
        return

    print(f"Removing {len(corrupted_files)} corrupted images...")
    for file_info in corrupted_files:
        try:
            os.remove(file_info['path'])
            print(f"✓ Removed: {file_info['path']}")
        except Exception as e:
            print(f"✗ Failed: {file_info['path']} — {e}")
    print("\n✓ Cleanup complete!")

# ========================================
# UNCOMMENT to remove corrupted files
# (requires working copy from Step 2)
# ========================================
# remove_corrupted_images(corrupted_files)

In [5]:
## Step 5: Standardize Image Format (Optional)

def standardize_image_format(base_path, target_format='jpg'):
    print(f"Standardizing images to .{target_format} format...\n")
    converted_count = 0

    for split in ['train', 'test']:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            continue

        classes = [d for d in os.listdir(split_path)
                  if os.path.isdir(os.path.join(split_path, d))]

        for class_name in tqdm(classes, desc=f"Converting {split}"):
            class_path = os.path.join(split_path, class_name)
            images = [f for f in os.listdir(class_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            for img_name in images:
                if img_name.lower().endswith(f'.{target_format}'):
                    continue
                img_path = os.path.join(class_path, img_name)
                try:
                    img = Image.open(img_path)
                    if img.mode != 'RGB':
                        img = img.convert('RGB')
                    new_name = os.path.splitext(img_name)[0] + f'.{target_format}'
                    img.save(os.path.join(class_path, new_name), format=target_format.upper())
                    os.remove(img_path)
                    converted_count += 1
                except Exception as e:
                    print(f"Error converting {img_path}: {e}")

    print(f"\n✓ Converted {converted_count} images to .{target_format}")

# ========================================
# UNCOMMENT to standardize format
# ========================================
# standardize_image_format(DATASET_PATH, target_format='jpg')

In [6]:
## Step 6: Check for Duplicate Images

from collections import defaultdict
import hashlib

def find_duplicate_images(base_path, check_content=False):
    print("Checking for duplicate images...\n")
    duplicates = defaultdict(list)

    for split in ['train', 'test']:
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            continue

        classes = [d for d in os.listdir(split_path)
                  if os.path.isdir(os.path.join(split_path, d))]

        for class_name in tqdm(classes, desc=f"Checking {split}"):
            class_path = os.path.join(split_path, class_name)
            images = [f for f in os.listdir(class_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

            for img_name in images:
                img_path = os.path.join(class_path, img_name)
                if check_content:
                    with open(img_path, 'rb') as f:
                        key = hashlib.md5(f.read()).hexdigest()
                else:
                    key = img_name
                duplicates[key].append(img_path)

    actual_duplicates = {k: v for k, v in duplicates.items() if len(v) > 1}

    if actual_duplicates:
        print(f"⚠️  Found {len(actual_duplicates)} sets of duplicate images:")
        for i, (key, paths) in enumerate(list(actual_duplicates.items())[:10], 1):
            print(f"\n   Set {i}:")
            for p in paths:
                print(f"      • {p}")
    else:
        print("✓ No duplicate images found!")

    return actual_duplicates

duplicates = find_duplicate_images(DATASET_PATH, check_content=False)

Checking for duplicate images...



Checking train:   0%|          | 0/23 [00:00<?, ?it/s]

Checking test:   0%|          | 0/23 [00:00<?, ?it/s]

⚠️  Found 698 sets of duplicate images:

   Set 1:
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Light Diseases and Disorders of Pigmentation/mongolian-spot-1.jpg
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Melanoma Skin Cancer Nevi and Moles/mongolian-spot-1.jpg

   Set 2:
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Light Diseases and Disorders of Pigmentation/milia-15.jpg
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Acne and Rosacea Photos/milia-15.jpg

   Set 3:
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Light Diseases and Disorders of Pigmentation/milia-10.jpg
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Acne and Rosacea Photos/milia-10.jpg

   Set 4:
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Light Diseases and Disorders of Pigmentation/milia-16.jpg
      • /kaggle/input/datasets/shubhamgoel27/dermnet/train/Acne and Rosacea Photos/milia-16.jpg

   Set 5:
      • /kaggle/input/dat

In [7]:
## Step 7: Generate Cleaning Report

print("\n" + "="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"\n✓ Corrupted Images Found: {len(corrupted_files)}")
if corrupted_files:
    print("   (Copy dataset to working dir then uncomment removal function)")
print(f"\n✓ Duplicate Sets Found: {len(duplicates)}")
print("\n✓ Data cleaning check complete!")
print("\n" + "="*60)


DATA CLEANING SUMMARY

✓ Corrupted Images Found: 0

✓ Duplicate Sets Found: 698

✓ Data cleaning check complete!



In [8]:
## Summary

##✅ Checked for corrupted images  
##✅ Checked for duplicate images  
##✅ Cleaning report generated  

##⚠️ To actually modify files, copy dataset to `/kaggle/working/dermnet_working/` first.